# Hands-on: A 1D CNN that Finds DNA Motifs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/DNA_motif_colab.ipynb)

A convolutional filter does not have to slide over an image. Here it slides **along a DNA sequence**,
looking for a short pattern (a **motif**), such as a binding site. This is how CNNs are used in genomics,
for example to find where proteins bind to DNA.

The task is synthetic so that we know the right answer: some sequences contain a hidden motif (with a few
random mutations), and the network must learn to tell which ones. Then we check whether the filter it
learned **recovers the motif on its own**.

**No coding needed:** run each cell in order with the ▶ button (or `Shift + Enter`). Cells marked
**✏️ Settings** have values you can change. This notebook runs in seconds, even without a GPU.

## 1. ✏️ Settings

- `MOTIF`: the hidden pattern (letters A, C, G, T). `TATAAT` is the bacterial *Pribnow box*, part of a
  promoter. Try your own!
- `SEQ_LEN`: length of each sequence. `N_SEQ`: number of sequences (half contain the motif).
- `MUTATION_RATE`: chance that each letter of the planted motif is changed to a random letter.
- `N_FILTERS`, `KERNEL_SIZE`: number and length of the 1D filters.

In [ ]:
MOTIF = "TATAAT"
SEQ_LEN = 50
N_SEQ = 4000
MUTATION_RATE = 0.15
N_FILTERS = 4
KERNEL_SIZE = 8
EPOCHS = 30

## 2. Create the data

Each sequence is turned into a **one-hot** tensor with shape `(4, L)`: one row per letter (A, C, G, T)
and one column per position, as in the Session 1 slide on tensors. Positive sequences have the motif
planted at a random position; negative sequences are purely random.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

plt.rcParams.update({"font.size": 14, "lines.linewidth": 3})
BASES = "ACGT"
rng = np.random.default_rng(0)
torch.manual_seed(0)

def make_sequence(with_motif):
    seq = list(rng.choice(list(BASES), SEQ_LEN))
    pos = -1
    if with_motif:
        pos = int(rng.integers(0, SEQ_LEN - len(MOTIF) + 1))
        for k, base in enumerate(MOTIF):
            seq[pos + k] = rng.choice(list(BASES)) if rng.random() < MUTATION_RATE else base
    seq = "".join(seq)
    if not with_motif and MOTIF in seq:  # keep negatives truly motif-free
        return make_sequence(False)
    return seq, pos

def one_hot(seq):
    x = np.zeros((4, len(seq)), dtype=np.float32)
    x[[BASES.index(b) for b in seq], np.arange(len(seq))] = 1
    return x

labels = np.array([1] * (N_SEQ // 2) + [0] * (N_SEQ // 2))
rng.shuffle(labels)
seqs, positions = zip(*[make_sequence(bool(l)) for l in labels])
X = torch.from_numpy(np.stack([one_hot(s) for s in seqs]))
y = torch.from_numpy(labels)

n_test = N_SEQ // 5
X_train, y_train, X_test, y_test = X[n_test:], y[n_test:], X[:n_test], y[:n_test]
print("Tensor shape:", tuple(X.shape), "(sequences, letters, positions)")
print(f"Training: {len(y_train)} sequences, test: {len(y_test)}\n")
for i in np.flatnonzero(labels)[:3]:
    p = positions[i]
    s = seqs[i]
    print(f"with motif: ...{s[max(0, p - 15):p]}[{s[p:p + len(MOTIF)]}]{s[p + len(MOTIF):p + len(MOTIF) + 15]}...")

## 3. Baseline: exact search

Before using a neural network, try the simplest rule: "the sequence contains exactly the motif". It
fails whenever a mutation changed one of the motif's letters. Chance level is 50%.

In [ ]:
test_seqs = seqs[:n_test]
exact = np.array([MOTIF in s for s in test_seqs])
baseline_acc = (exact == y_test.numpy()).mean()
print(f"Exact-search accuracy on the test set: {baseline_acc:.1%}")

## 4. Train the 1D CNN

The model has a single convolutional layer: each filter slides along the sequence and gives a score at
every position. **Global max pooling** keeps only the best score of each filter ("was the pattern found
anywhere?"), and a linear layer makes the decision.

In [ ]:
class MotifCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(4, N_FILTERS, KERNEL_SIZE)
        self.classifier = nn.Linear(N_FILTERS, 2)

    def activations(self, x):
        return torch.relu(self.conv(x))  # (batch, filters, positions)

    def forward(self, x):
        return self.classifier(self.activations(x).amax(dim=2))

model = MotifCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(EPOCHS):
    model.train()
    for idx in torch.randperm(len(y_train)).split(64):
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(X_train[idx]), y_train[idx])
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    cnn_acc = (model(X_test).argmax(1) == y_test).float().mean().item()
print(f"Chance:        50.0%")
print(f"Exact search:  {baseline_acc:.1%}")
print(f"1D CNN:        {cnn_acc:.1%}")

## 5. What did the filter learn?

We pick the filter that the classifier relies on most for "motif present" and draw its weights in two
ways: a **heatmap** (rows A, C, G, T; columns are positions within the filter) and a **sequence logo**,
where taller letters mean the filter likes that letter more at that position. Can you read your motif?

In [ ]:
from matplotlib.textpath import TextPath
from matplotlib.patches import PathPatch
from matplotlib.font_manager import FontProperties
from matplotlib.transforms import Affine2D

COLORS = {"A": "#009E73", "C": "#0072B2", "G": "#E69F00", "T": "#D55E00"}
FONT = FontProperties(family="DejaVu Sans", weight="bold")

def draw_letter(ax, letter, x, y, height):
    path = TextPath((0, 0), letter, size=1, prop=FONT)
    box = path.get_extents()
    t = Affine2D().translate(-box.x0, -box.y0).scale(0.9 / box.width, height / box.height).translate(x + 0.05, y)
    ax.add_patch(PathPatch(t.transform_path(path), color=COLORS[letter], lw=0))

def draw_logo(ax, weights):
    heights = np.clip(weights, 0, None)
    for pos in range(heights.shape[1]):
        y0 = 0.0
        for b in np.argsort(heights[:, pos]):
            if heights[b, pos] > 0:
                draw_letter(ax, BASES[b], pos, y0, heights[b, pos])
                y0 += heights[b, pos]
    ax.set_xlim(0, heights.shape[1])
    ax.set_ylim(0, heights.sum(0).max() * 1.05)
    ax.set_xticks(np.arange(heights.shape[1]) + 0.5, range(1, heights.shape[1] + 1))
    ax.set_xlabel("Position in filter")
    ax.set_ylabel("Weight")

best = int((model.classifier.weight[1] - model.classifier.weight[0]).argmax())
W = model.conv.weight[best].detach().numpy()  # (4, KERNEL_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
lim = np.abs(W).max()
axes[0].imshow(W, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
axes[0].set_yticks(range(4), list(BASES))
axes[0].set_xticks(range(KERNEL_SIZE), range(1, KERNEL_SIZE + 1))
axes[0].set(xlabel="Position in filter", title=f"Filter {best + 1}: weights")
draw_logo(axes[1], W)
axes[1].set_title("Sequence logo")
plt.tight_layout()
plt.show()
print("Strongest letter at each position:", "".join(BASES[i] for i in W.argmax(0)), "| planted motif:", MOTIF)

## 6. Where in the sequence does the filter fire?

The filter's score at every position of one test sequence that contains the motif. The shaded band marks
where the motif was planted.

In [ ]:
i = int(np.flatnonzero(y_test.numpy() == 1)[0])
with torch.no_grad():
    track = model.activations(X_test[i:i + 1])[0, best].numpy()
p = positions[i]

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(np.arange(len(track)), track, color="#0072B2")
ax.axvspan(p - 0.5, p + len(MOTIF) - 0.5, color="#E69F00", alpha=0.3, label="Planted motif")
ax.set(xlabel="Position in sequence", ylabel="Filter response", title=f"Sequence {i}: {seqs[i][p:p + len(MOTIF)]} at position {p}")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## What to notice

- The same idea as image CNNs: a **small filter sliding along the data**, now in 1D along a sequence.
- The network was never told the motif, yet its filter **recovers it from labeled examples**, and it
  handles mutations that break an exact search.
- The activation track shows **where** the motif is, which makes the model easier to interpret.

**Try:** raise `MUTATION_RATE` to 0.3, use longer sequences (`SEQ_LEN = 200`), plant a longer motif, or make `KERNEL_SIZE` shorter than the motif.

## Questions for discussion

1. Real motifs (for example, transcription-factor binding sites) are not planted by us. How could you
   check whether a filter learned from real genomic data corresponds to a known motif?
2. What happens if the label depends on **two** motifs far apart in the sequence? Can one small filter
   capture that? (We will come back to this with transformers.)